# 05 — Advanced NumPy + Pandas Analysis

Practise IQR outliers, rolling metrics, pivot tables, joins, and validation. Interpret outlier flags as review candidates, not guaranteed errors.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'sales_cleaned.csv', parse_dates=['Order_Date'])
df['Net_Sales'] = df['Quantity'] * df['Unit_Price'] * (1 - df['Discount'])

## NumPy IQR outlier detection

In [ ]:
values = df['Net_Sales'].to_numpy(dtype=float)
q1, q3 = np.percentile(values, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df['Potential_Outlier'] = np.where((values < lower) | (values > upper), True, False)
print({'Q1': q1, 'Q3': q3, 'IQR': iqr, 'lower_fence': lower, 'upper_fence': upper})
df.loc[df['Potential_Outlier']].head(10)

## Pivot tables and reshaping

In [ ]:
pivot = pd.pivot_table(df, index='Region', columns='Category', values='Net_Sales', aggfunc='sum', fill_value=0, margins=True)
print(pivot)
long = pivot.reset_index().melt(id_vars='Region', var_name='Category', value_name='Revenue')
long.head()

## Rolling monthly average

In [ ]:
monthly = df.set_index('Order_Date')['Net_Sales'].resample('ME').sum().to_frame('Revenue')
monthly['Three_Month_Rolling_Average'] = monthly['Revenue'].rolling(3, min_periods=1).mean()
monthly

## Merge and validate customer information

In [ ]:
customer = df.groupby('Customer_ID', as_index=False).agg(Total_Revenue=('Net_Sales', 'sum'), Orders=('Order_ID', 'nunique'))
region_counts = df.groupby('Customer_ID')['Region'].nunique().rename('Region_Count').reset_index()
merged = customer.merge(region_counts, on='Customer_ID', how='left', validate='one_to_one', indicator=True)
print(merged['_merge'].value_counts())
merged.sort_values('Total_Revenue', ascending=False).head(10)

## Final challenges
1. Add a category contribution percentage to total revenue.
2. Compare mean and median transaction values by product.
3. Identify repeat customers with more than one distinct order.
4. Add a data-quality report before and after cleaning.
5. Refactor repeated code into functions under `src/`.
6. Write limitations and recommendations in `reports/findings.md`.